<a href="https://colab.research.google.com/github/swathi22101997/capstone-project/blob/main/capstone_project_module3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 668.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found e

In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. Define document contents (8 policy documents)
# Replace/extend these with the actual text of your 8 policy documents or load them from files
DOCUMENTS = [
    {
        "id": "doc_1",
        "title": "Return and Refund Policy",
        "text": "Zepto allows returns within 7 days of delivery for damaged or incorrect items. Refunds are processed to the original payment method within 5-7 business days after approval."
    },
    {
        "id": "doc_2",
        "title": "Delivery Terms and SLA",
        "text": "Standard delivery is targeted within 10 to 15 minutes depending on traffic and weather conditions. Orders above INR 99 qualify for free delivery."
    },
    {
        "id": "doc_3",
        "title": "Cancellation Policy",
        "text": "Orders can be canceled free of charge before the delivery partner is assigned. Once dispatched, a cancellation fee of up to INR 50 may apply."
    },
    {
        "id": "doc_4",
        "title": "Payment and Wallet Policy",
        "text": "We accept UPI, Credit/Debit cards, Net Banking, Cash on Delivery, and Zepto Cash wallet payments. Wallet balances are non-transferable to bank accounts."
    },
    {
        "id": "doc_5",
        "title": "Customer Support Operating Hours",
        "text": "Zepto customer support operates 24/7 via in-app chat. Email queries are responded to within 24 hours at support@zepto.com."
    },
    {
        "id": "doc_6",
        "title": "Quality and Freshness Guarantee",
        "text": "All fresh produce and dairy items undergo quality checks. If you receive spoiled items, raise a request in-app within 2 hours of delivery for an instant credit."
    },
    {
        "id": "doc_7",
        "title": "Account Safety and Privacy Policy",
        "text": "Zepto uses industry-standard encryption for user data and payment information. We never share personal contact details with third-party vendors."
    },
    {
        "id": "doc_8",
        "title": "Promotions and Coupon Terms",
        "text": "Promotional coupons cannot be combined on a single order. Coupons have individual minimum order value requirements and expiration dates."
    }
]

def chunk_document(doc, max_length=300):
    """
    Per-document or basic chunking scheme.
    If document text is within limits, returns a single chunk;
    otherwise splits into smaller chunks.
    """
    text = doc["text"]
    if len(text) <= max_length:
        return [{
            "chunk_id": f"{doc['id']}_c0",
            "text": text,
            "metadata": {"doc_id": doc["id"], "title": doc["title"], "chunk_index": 0}
        }]

    # Simple chunking by paragraph or fixed length if needed
    words = text.split()
    chunks = []
    chunk_size = 50  # words per chunk
    for i in range(0, len(words), chunk_size):
        chunk_text = " ".join(words[i:i + chunk_size])
        chunks.append({
            "chunk_id": f"{doc['id']}_c{i // chunk_size}",
            "text": chunk_text,
            "metadata": {"doc_id": doc["id"], "title": doc["title"], "chunk_index": i // chunk_size}
        })
    return chunks

def build_vector_store():
    # 2. Initialize ChromaDB persistent client
    client = chromadb.PersistentClient(path="./chroma_db")

    # 3. Use SentenceTransformer embedding function with 'all-MiniLM-L6-v2'
    embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )

    # 4. Get or create collection
    collection = client.get_or_create_collection(
        name="zepto_policies",
        embedding_function=embedding_fn,
        metadata={"hnsw:space": "cosine"}
    )

    # 5. Process and chunk all 8 documents
    ids = []
    documents = []
    metadatas = []

    for doc in DOCUMENTS:
        chunks = chunk_document(doc)
        for chunk in chunks:
            ids.append(chunk["chunk_id"])
            documents.append(chunk["text"])
            metadatas.append(chunk["metadata"])

    # 6. Add embeddings to ChromaDB
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )

    print(f"Successfully embedded and indexed {len(documents)} chunks across 8 documents in ChromaDB collection 'zepto_policies'.")

if __name__ == "__main__":
    build_vector_store()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Successfully embedded and indexed 8 chunks across 8 documents in ChromaDB collection 'zepto_policies'.


[ROLE]
You are a precise, professional customer support AI assistant for Zepto. Your sole purpose is to answer user queries accurately based strictly on the provided company policy documents.

[CONTEXT]
{retrieved_documents}

[TASK]
1. Read the user's question and analyze the provided [CONTEXT] documents.
2. Formulate a direct, accurate answer using ONLY details explicitly mentioned in the [CONTEXT].
3. If the provided context does not contain enough information to answer the question, state clearly that you do not have that information available.

[NEGATIVE CONSTRAINTS]
- DO NOT answer using any external knowledge, assumptions, or information not present in the provided context.
- DO NOT invent, hallucinate, or infer policies or timeframes that are not explicitly written above.
- DO NOT make promises or suggest solutions outside the scope of the context.

[FORMAT]
- Deliver the answer directly without introductory conversational fluff (e.g., avoid "Based on the documents provided...").
- Use clear, bulleted key points for multi-part details or step-by-step conditions.
- Embed document/policy titles in bold where relevant.

[LENGTH]
Keep the response concise and strictly focused on the query—maximum of 3 to 4 sentences or bullet points (under 80 words).

---

[FEW-SHOT EXAMPLES]

Example 1:
Context:
Document: "Return and Refund Policy" - Zepto allows returns within 7 days of delivery for damaged or incorrect items. Refunds are processed to the original payment method within 5-7 business days after approval.

User Query:
How long do I have to return a broken item, and how will I get my money back?

Response:
- You can request a return within **7 days of delivery** for damaged items.
- Once approved, refunds are processed back to your **original payment method** within **5-7 business days**.

Example 2:
Context:
Document: "Delivery Terms and SLA" - Standard delivery is targeted within 10 to 15 minutes depending on traffic and weather conditions. Orders above INR 99 qualify for free delivery.

User Query:
Can I change my delivery address after placing an order?

Response:
I do not have information regarding address changes after placing an order in the provided policy documents. Please reach out to customer support via the app for further assistance.

---

[USER QUERY]
{user_query}

[RESPONSE]

In [ ]:
import os
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END, START

# ---------------------------------------------------------------------------
# Toggle configuration
# MOCK_LLM = "1" (or unset) -> Uses baseline heuristic (no API call needed)
# MOCK_LLM = "0"            -> Calls real LLM via LangChain / Gemini / OpenAI
# ---------------------------------------------------------------------------
MOCK_LLM = os.getenv("MOCK_LLM", "1")

# Optional: Import real LLM if MOCK_LLM=0
if MOCK_LLM == "0":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


# ===========================================================================
# 1. State Definition
# ===========================================================================
class GraphState(TypedDict):
    query: str
    intent: Optional[str]               # "policy_question" or "general_question"
    retrieved_docs: Optional[List[str]] # Documents populated if retrieval needed
    response: Optional[str]             # Final generated output


# ===========================================================================
# 2. Node Definitions
# ===========================================================================

def classify_intent(state: GraphState) -> GraphState:
    """Node 1: Classifies the query as 'policy_question' or 'general_question'."""
    query = state["query"]

    if os.getenv("MOCK_LLM", "1") != "0":
        # Mock / Baseline: Keyword-based heuristic
        keywords = ["delivery", "return", "refund", "membership", "tracking", "cancel", "gift card", "support hours"]
        if any(keyword in query.lower() for keyword in keywords):
            intent = "policy_question"
        else:
            intent = "general_question"
    else:
        # Real LLM Extension: Zero-shot classification
        prompt = f"Classify this query as either 'policy_question' or 'general_question'. Respond with ONLY the exact category string.\nQuery: {query}"
        intent = llm.invoke(prompt).content.strip().lower()

    return {**state, "intent": intent}


def retrieve_or_pass(state: GraphState) -> GraphState:
    """Node 2: Fetches policy context if needed, or passes through for general queries."""
    intent = state.get("intent")
    query = state["query"]

    if intent == "policy_question":
        if os.getenv("MOCK_LLM", "1") != "0":
            # Mock / Baseline: Simulated vector database retrieval
            docs = [
                "Zepto Policy Doc: Returns are permitted within 7 days of delivery for damaged goods.",
                "Zepto Policy Doc: Standard delivery takes 10-15 minutes subject to operational conditions."
            ]
        else:
            # Real LLM Extension: Real VectorDB / ChromaDB retrieval call would go here
            docs = ["Retrieved dynamic context from ChromaDB index for Zepto policies."]
    else:
        docs = []

    return {**state, "retrieved_docs": docs}


def generate_response(state: GraphState) -> GraphState:
    """Node 3: Generates the final answer using retrieved context or direct response."""
    query = state["query"]
    intent = state.get("intent")
    docs = state.get("retrieved_docs", [])

    if os.getenv("MOCK_LLM", "1") != "0":
        # Mock / Baseline: Template response generation
        if intent == "policy_question":
            context_str = " ".join(docs)
            response = f"[Mock Policy Output] Based on context ('{context_str}'): We have answered your query about '{query}'."
        else:
            response = f"[Mock General Output] Thank you for your question: '{query}'. How else can I assist you today?"
    else:
        # Real LLM Extension: Standard RAG or direct query prompt execution
        if intent == "policy_question":
            prompt = f"Context: {' '.join(docs)}\n\nUser Question: {query}\n\nAnswer the question using the context above:"
        else:
            prompt = f"Answer the user query in a friendly customer support tone:\nQuestion: {query}"

        response = llm.invoke(prompt).content

    return {**state, "response": response}


# ===========================================================================
# 3. LangGraph Workflow Assembly
# ===========================================================================

builder = StateGraph(GraphState)

# Add all 3 nodes
builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_or_pass", retrieve_or_pass)
builder.add_node("generate_response", generate_response)

# Connect workflow graph edges linearly
builder.add_edge(START, "classify_intent")
builder.add_edge("classify_intent", "retrieve_or_pass")
builder.add_edge("retrieve_or_pass", "generate_response")
builder.add_edge("generate_response", END)

# Compile graph
graph = builder.compile()


# ===========================================================================
# 4. Example Execution
# ===========================================================================

if __name__ == "__main__":
    # Test Policy Query
    policy_input: GraphState = {"query": "What is your refund policy?", "intent": None, "retrieved_docs": None, "response": None}
    policy_result = graph.invoke(policy_input)
    print("--- Policy Query Result ---")
    print(f"Intent: {policy_result['intent']}")
    print(f"Docs: {policy_result['retrieved_docs']}")
    print(f"Response: {policy_result['response']}\n")

    # Test General Query
    general_input: GraphState = {"query": "Hello, hope you are having a nice day!", "intent": None, "retrieved_docs": None, "response": None}
    general_result = graph.invoke(general_input)
    print("--- General Query Result ---")
    print(f"Intent: {general_result['intent']}")
    print(f"Docs: {general_result['retrieved_docs']}")
    print(f"Response: {general_result['response']}")

--- Policy Query Result ---
Intent: policy_question
Docs: ['Zepto Policy Doc: Returns are permitted within 7 days of delivery for damaged goods.', 'Zepto Policy Doc: Standard delivery takes 10-15 minutes subject to operational conditions.']
Response: [Mock Policy Output] Based on context ('Zepto Policy Doc: Returns are permitted within 7 days of delivery for damaged goods. Zepto Policy Doc: Standard delivery takes 10-15 minutes subject to operational conditions.'): We have answered your query about 'What is your refund policy?'.

--- General Query Result ---
Intent: general_question
Docs: []
Response: [Mock General Output] Thank you for your question: 'Hello, hope you are having a nice day!'. How else can I assist you today?


In [ ]:
import os
from typing import TypedDict, List, Optional, Literal
from langgraph.graph import StateGraph, END, START

# ---------------------------------------------------------------------------
# Configuration & State Definition
# ---------------------------------------------------------------------------
MOCK_LLM = os.getenv("MOCK_LLM", "1")

if MOCK_LLM == "0":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


class GraphState(TypedDict):
    query: str
    intent: Optional[str]
    retrieved_docs: Optional[List[str]]
    response: Optional[str]


# ===========================================================================
# Node Definitions
# ===========================================================================

def classify_intent(state: GraphState) -> GraphState:
    """Node 1: Classifies the query as 'policy_question' or 'general_question'."""
    query = state["query"]

    if os.getenv("MOCK_LLM", "1") != "0":
        # Mock / Baseline: Keyword-based heuristic
        keywords = ["delivery", "return", "refund", "membership", "tracking", "cancel", "gift card", "support hours"]
        if any(keyword in query.lower() for keyword in keywords):
            intent = "policy_question"
        else:
            intent = "general_question"
    else:
        # Real LLM Extension: Zero-shot classification
        prompt = f"Classify this query as either 'policy_question' or 'general_question'. Respond with ONLY the exact category string.\nQuery: {query}"
        intent = llm.invoke(prompt).content.strip().lower()

    return {**state, "intent": intent}


def direct_answer(state: GraphState) -> GraphState:
    """
    Node for handling 'general_question' queries:
    - Mock Mode (MOCK_LLM=1 or unset): Returns a fixed canned string without calling an LLM.
    - Real LLM Extension (MOCK_LLM=0): Prompts the LLM directly without performing context retrieval.
    """
    query = state["query"]

    if os.getenv("MOCK_LLM", "1") != "0":
        # Mock Mode (Graded Baseline): Fixed canned response
        response = "I can only answer questions about Zepto policies right now."
    else:
        # Real LLM Extension (MOCK_LLM=0): Direct LLM call without retrieval
        prompt = f"Answer the following user query in a friendly customer support tone:\nQuestion: {query}"
        response = llm.invoke(prompt).content

    return {
        **state,
        "retrieved_docs": [],
        "response": response
    }

def retrieve_and_answer(state: GraphState) -> GraphState:
    """
    Node for handling 'policy_question' queries:
    - Retrieves relevant documents.
    - Generates an answer based on the retrieved context.
    """
    query = state["query"]
    # intent is guaranteed to be 'policy_question' due to routing

    # Retrieval logic (from original retrieve_or_pass for policy questions)
    if os.getenv("MOCK_LLM", "1") != "0":
        docs = [
            "Zepto Policy Doc: Returns are permitted within 7 days of delivery for damaged goods.",
            "Zepto Policy Doc: Standard delivery takes 10-15 minutes subject to operational conditions."
        ]
    else:
        # This is where actual ChromaDB retrieval would go
        docs = ["Retrieved dynamic context from ChromaDB index for Zepto policies."]

    # Generate response logic (from original generate_response for policy questions)
    if os.getenv("MOCK_LLM", "1") != "0":
        context_str = " ".join(docs)
        response = f"[Mock Policy Output] Based on context ('{context_str}'): We have answered your query about '{query}'."
    else:
        prompt = f"Context: {' '.join(docs)}\n\nUser Question: {query}\n\nAnswer the question using the context above:"
        response = llm.invoke(prompt).content

    return {
        **state,
        "retrieved_docs": docs,
        "response": response
    }


# ===========================================================================
# Conditional Router Function
# ===========================================================================
def route_by_intent(state: GraphState) -> Literal["retrieve_and_answer", "direct_answer"]:
    """
    Conditional routing function that reads state['intent']
    and determines the next target node.

    This routing logic operates independently of MOCK_LLM.
    """
    if state.get("intent") == "policy_question":
        return "retrieve_and_answer"
    return "direct_answer"


# ===========================================================================
# LangGraph Workflow Wiring with Conditional Edges
# ===========================================================================

builder = StateGraph(GraphState)

# Add Nodes
builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_and_answer", retrieve_and_answer)
builder.add_node("direct_answer", direct_answer)

# Flow Setup: START -> classify_intent
builder.add_edge(START, "classify_intent")

# Conditional Edge Routing from classify_intent based on intent
builder.add_conditional_edges(
    "classify_intent",
    route_by_intent,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

# Connect branch terminal nodes to END
builder.add_edge("retrieve_and_answer", END)
builder.add_edge("direct_answer", END)

# Compile the Graph
graph = builder.compile()

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError
import json

# ---------------------------------------------------------------------------
# 1. Define the Structured Output Schema
# ---------------------------------------------------------------------------
class FinalAnswer(BaseModel):
    answer: str = Field(
        description="The final response text to the user query."
    )
    sources: List[str] = Field(
        default_factory=list,
        description="List of document/chunk IDs referenced (empty for general questions)."
    )
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence score between 0.0 and 1.0."
    )

# ---------------------------------------------------------------------------
# 2. Node Implementation (Supports Mock & Real LLM with Retries)
# ---------------------------------------------------------------------------
def generate_structured_answer(state: dict, llm_client=None, mock_mode: bool = True) -> dict:
    query_type = state.get("query_type", "general_question")
    retrieved_chunks = state.get("retrieved_chunks", [])  # List of dicts with 'chunk_id'

    # Extract sources if present
    source_ids = [chunk["chunk_id"] for chunk in retrieved_chunks if "chunk_id" in chunk]

    # --- MODE 1: MOCK MODE ---
    if mock_mode or llm_client is None:
        if query_type == "policy_question":
            mock_payload = FinalAnswer(
                answer="Zepto allows returns within 7 days of delivery for damaged or incorrect items.",
                sources=source_ids if source_ids else ["doc_1_c0"],
                confidence=1.0
            )
        else:
            mock_payload = FinalAnswer(
                answer="I am an AI assistant designed to help answer questions about store policies.",
                sources=[],
                confidence=1.0
            )
        return {"final_output": mock_payload.model_dump()}

    # --- MODE 2: REAL LLM MODE (MOCK_LLM=0) WITH RETRIES ---
    structured_llm = llm_client.with_structured_output(FinalAnswer)

    context = "\n".join([c.get("text", "") for c in retrieved_chunks])
    system_prompt = (
        "You are a helpful customer service assistant. "
        "Provide a clear answer based on the provided context. "
        "Include all source chunk IDs used. Output MUST strictly match the required JSON schema."
    )

    user_prompt = f"Query: {state['query']}\nContext: {context}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    max_retries = 2
    for attempt in range(max_retries + 1):
        try:
            # Call LLM with structured output enforcement
            response = structured_llm.invoke(messages)

            # Verify validity via Pydantic model
            if isinstance(response, FinalAnswer):
                validated_output = response
            else:
                validated_output = FinalAnswer.model_validate(response)

            return {"final_output": validated_output.model_dump()}

        except (ValidationError, Exception) as e:
            if attempt < max_retries:
                # Append corrective instruction for the retry prompt
                corrective_msg = (
                    f"Your previous response failed validation with error: {str(e)}. "
                    "Please regenerate the response ensuring strictly valid JSON "
                    "matching fields: 'answer' (str), 'sources' (list of strings), and 'confidence' (float 0.0-1.0)."
                )
                messages.append({"role": "user", "content": corrective_msg})
            else:
                # Standard error response upon reaching maximum retries
                fallback_error = {
                    "error": "Failed to generate output matching required schema after retries.",
                    "details": str(e)
                }
                return {"final_output": fallback_error}

In [ ]:
import os
from typing import List
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import uvicorn
import nest_asyncio
import threading # Import threading

nest_asyncio.apply()

# Import or define your Pydantic schemas and state/graph logic here
class AskRequest(BaseModel):
    query: str = Field(..., example="What is the return policy for damaged items?")

class FinalAnswer(BaseModel):
    answer: str = Field(description="The final response text to the user query.")
    sources: List[str] = Field(
        default_factory=list,
        description="List of document/chunk IDs referenced (empty for general questions)."
    )
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence score between 0.0 and 1.0."
    )

# Instantiate FastAPI app
app = FastAPI(
    title="Customer Service Graph API",
    description="FastAPI wrapper for LangGraph RAG system",
    version="1.0.0"
)

# Example graph runner logic (integrating mock/graph mode)
def run_graph(query: str, mock_mode: bool = True) -> FinalAnswer:
    # Read environment variable default if set
    mock_mode = os.getenv("MOCK_LLM", "1") == "1"

    # Simple classification heuristic for demonstration
    policy_keywords = ["return", "refund", "delivery", "cancel", "payment", "hours", "quality"]
    is_policy_query = any(keyword in query.lower() for keyword in policy_keywords)

    if is_policy_query:
        # Represents retrieval-triggered state output
        return FinalAnswer(
            answer="Zepto allows returns within 7 days of delivery for damaged or incorrect items. Refunds are processed within 5-7 business days.",
            sources=["doc_1_c0"],
            confidence=1.0
        )
    else:
        # Represents general question state output (no retrieval)
        return FinalAnswer(
            answer="I am an AI assistant designed to help answer questions about store policies and orders.",
            sources=[],
            confidence=1.0
        )

@app.post("/ask", response_model=FinalAnswer)
async def ask_endpoint(request: AskRequest):
    try:
        # Run graph process with user query
        result = run_graph(query=request.query)
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    # Function to run uvicorn server
    def run_uvicorn_server():
        uvicorn.run(app, host="0.0.0.0", port=8000)

    # Start the uvicorn server in a separate thread
    server_thread = threading.Thread(target=run_uvicorn_server)
    server_thread.daemon = True  # Allows the main program to exit even if the thread is still running
    server_thread.start()

    print("FastAPI app is starting in a background thread on http://0.0.0.0:8000")
    print("You will need to expose this port (e.g., via ngrok) to access it externally.")

/tmp/ipykernel_965/1138115792.py:13: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  query: str = Field(..., example="What is the return policy for damaged items?")


FastAPI app is starting in a background thread on http://0.0.0.0:8000
You will need to expose this port (e.g., via ngrok) to access it externally.


INFO:     Started server process [965]


You're right, placing Dockerfile syntax directly in a code cell will result in a `SyntaxError` because Colab cells execute Python by default. A Dockerfile is a special text file that contains instructions for building a Docker image, and it's not meant to be run as a Python script.

To create a Dockerfile within your Colab environment, you can write its content to a file named `Dockerfile` using Python's file operations. Here's how you can do it:

In [12]:
%%writefile Dockerfile
FROM python:3.10-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    curl \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .

RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]

Writing Dockerfile


Here is the short architecture section formatted for your **`README.md`**:

```markdown
## 🏗️ RAG Pipeline Architecture

This section details the end-to-end data flow and component responsibilities across all four stages of the RAG pipeline.


```

┌─────────────────────────────────────────────────────────────────────────────┐
│ 1. INGESTION & EMBEDDING (Setup / Startup)                                  │
│                                                                             │
│ Policy Documents ──> chunk_document() ──> SentenceTransformerEmbeddingFunction│
│                                                     │                       │
│                                                     ▼                       │
│                                           ChromaDB Collection               │
│                                            ("zepto_policies")              │
└─────────────────────────────────────────────────────────────────────────────┘
│
┌─────────────────────────────────────────────────────┼───────────────────────┐
│ 2 & 3. RETRIEVAL & GENERATION (Per-Request Flow)   │                       │
│                                                     │                       │
│  User Query (POST /ask) ──> [retrieve_node] ◄───────┘                       │
│                                  │                                          │
│                                  ▼                                          │
│                          [generate_node]                                    │
│                                  │                                          │
│                         ┌────────┴────────┐                                 │
│                         │ MOCK_LLM Toggle │                                 │
│                         └────────┬────────┘                                 │
│               True (Default)     │      False (Real LLM)                    │
│             ┌────────────────────┴────────────────────┐                     │
│             ▼                                         ▼                     │
│    Deterministic Mock                     Gemini API Call                   │
│     Structured Output                    Structured Output                  │
│             └────────────────────┬────────────────────┘                     │
│                                  ▼                                          │
│                          FinalAnswer Model ──> HTTP Response                │
└─────────────────────────────────────────────────────────────────────────────┘

```

---

### Pipeline Stages Walkthrough

1. **Ingestion Stage**
   * **Handling Component:** `chunk_document()` function in `main.py`.
   * **Process & Data Flow:** Takes raw policy document dictionaries (`id`, `title`, `text`) and splits them into smaller, structured text chunks with metadata (`doc_id`, `title`, `chunk_index`). If a document is under the length threshold, it is preserved as a single chunk.

2. **Embedding Stage**
   * **Handling Component:** `SentenceTransformerEmbeddingFunction` (using `all-MiniLM-L6-v2`) inside `build_vector_store()`.
   * **Storage Target:** Chunks and vector embeddings are stored in a local **ChromaDB** persistent collection named `"zepto_policies"`.

3. **Retrieval Stage**
   * **Handling Component:** `retrieve_node()` function in the LangGraph workflow.
   * **Process & Data Flow:** When a `POST /ask` request arrives, `retrieve_node` reads `state["query"]` and queries the `"zepto_policies"` ChromaDB collection for the top relevant chunks. It writes the matching text excerpts to `state["retrieved_docs"]` and populates `state["sources"]` with the source document IDs/titles.

4. **Generation Stage**
   * **Handling Component:** `generate_node()` function in the LangGraph workflow using a structured system prompt.
   * **Process & Data Flow:** Receives the context chunks and original query from the graph state, constructs the prompt instructing the system to cite sources strictly from context, and formats the output into the `FinalAnswer` Pydantic model (`answer: str`, `sources: List[str]`).

---

### 🔀 `MOCK_LLM` Toggle Branching

The **Generation Stage** is the only stage that branches based on the `MOCK_LLM` environment variable toggle:

* **Default / Mock State (`MOCK_LLM=True` / `1`):**
  * Bypasses external API calls completely and uses an internal deterministic function inside `generate_node`.
  * Returns a predefined structured `FinalAnswer` referencing the retrieved sources, enabling offline testing without requiring an API key or incurring quota costs.
* **Real-LLM State (`MOCK_LLM=False` / `0`):**
  * Uses the `GEMINI_API_KEY` from the environment to call the Gemini API via standard structured output parameters (`.with_structured_output(FinalAnswer)`).
  * Dynamically synthesizes the answer from the retrieved context chunks in ChromaDB while strictly enforcing the output schema.
* **Unchanged Stages:** The Ingestion, Embedding, and Retrieval stages operate identically using ChromaDB and `SentenceTransformers` regardless of the `MOCK_LLM` setting.

```